# LoanLens — Exploratory Data Analysis (EDA)

This notebook begins the data-understanding phase for **LoanLens**, an explainable loan-decision intelligence project. It intentionally stops before feature engineering and model training.

**Goal of this section:** confirm the dataset's structure, types, completeness, and basic integrity before making modelling decisions.

> EDA means asking *what the data contains and whether it is trustworthy enough to analyse* before asking a model to learn from it.

## 1. Imports and setup

We use `pandas` to load and inspect the tabular data. `Path` makes the file location work whether the notebook is launched from the repository root or the `notebooks/` folder.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Load the dataset

The raw CSV stays unchanged in `data/raw/`. Keeping raw data separate from later cleaned data makes the analysis reproducible and lets us trace every transformation.

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "loan_approval.csv"
assert data_path.exists(), f"Dataset not found: {data_path}"

df = pd.read_csv(data_path)
print(f"Loaded {data_path.name} with {df.shape[0]:,} rows and {df.shape[1]} columns.")

## 3. First look at the data

`shape` tells us the dataset size, while `head()` shows a small sample. This quick check is useful for spotting incorrect separators, shifted columns, or unexpected values immediately after loading.

In [ ]:
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns:")
print(df.columns.tolist())

df.head()

### Reading the columns

`Loan_Status` is the outcome we will eventually predict: `Approved` or `Rejected`. `Loan_ID` is an application identifier, so it should be retained for traceability but not treated as a meaningful predictive feature without further evidence.

At this stage, we only document the dataset. We do **not** change values, encode categories, fill missing data, or train a model.

## 4. Data types and feature groups

A column's data type affects how it should later be analysed and preprocessed. Numerical features can be summarized with statistics; categorical features need frequency-based inspection and later encoding.

In [ ]:
df.info()

numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()

print(f"\nNumerical columns ({len(numeric_columns)}): {numeric_columns}")
print(f"Categorical/text columns ({len(categorical_columns)}): {categorical_columns}")

## 5. Initial data-quality checks

Before interpreting patterns, we check for missing values, duplicate records, and identifier uniqueness. These checks prevent misleading summaries and reveal what needs careful handling later.

Missing values are reported as both a count and a percentage. We will choose any imputation strategy later, inside a train/test-safe modelling pipeline—not here.

In [ ]:
missing_values = df.isna().sum()
missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percent": (missing_values / len(df) * 100).round(2),
})

print("Missing-value summary:")
print(missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending=False).to_string())

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Unique Loan_ID values: {df['Loan_ID'].nunique():,} of {len(df):,} rows")
print(f"Rows with a missing Loan_ID: {df['Loan_ID'].isna().sum():,}")

### Initial quality conclusion

The missing-value report identifies a small number of incomplete fields to address later. Duplicate and identifier checks establish whether each record appears to represent a distinct application. We preserve this raw evidence rather than silently correcting it.

## 6. Target sanity check

Although modelling comes later, it is important to verify the classes in the target now. A strongly imbalanced target could make accuracy misleading and affect the evaluation plan.

In [ ]:
target_summary = (
    df["Loan_Status"]
    .value_counts(dropna=False)
    .rename_axis("Loan_Status")
    .reset_index(name="count")
)
target_summary["percentage"] = (target_summary["count"] / len(df) * 100).round(2)
target_summary

## What comes next

The next EDA section will examine feature distributions and relationships with `Loan_Status`, including a careful check of whether any feature has a suspiciously strong association with the target. Only after that evidence is documented will we design preprocessing and modelling.